# **Project — LMFP(LiMn_xFe₁₋ₓPO₄) 양극재: Mn 비율과 Mg 도핑이 충방전 부피 변화를 어떻게 바꾸나**

인공지능재료설계 · 개인 프로젝트 (난이도 상 — 포텐셜로 구조 최적화)

---

**설계 질문** — 올리빈 LiFePO₄ 의 Fe 를 Mn 으로 0 ~ 100 % 바꾸고(LMFP), Mg 를 25 % 넣으면 방전 ↔ 충전 부피 변화 ΔV 와 이론 용량이 어떻게 변하는가?

```
단계 1  [관문] LiFePO₄ 최적화 → 격자상수 재현 → Li 를 빼 FePO₄ 최적화 → ΔV          (Tutorial 09a 방식)
단계 2  MP 의 정답 — insertion_electrodes 의 max_delta_volume                           (Ch.6 방식)  [API key]
단계 3  [검증] 전부 치환 LiMPO₄ (M = Fe · Mn · Co · Ni) → 계산 ΔV vs MP 패리티           (09b 단계 4 방식)
참고    포텐셜 4종 비교 (v1 에서 계산한 표를 불러와 그림으로)
단계 4  [설계] 검증을 통과한 Fe–Mn 계만: Mn 0 · 25 · 50 · 75 · 100 % + Mg 25 %            (09b 단계 1 · 5 방식)
단계 5  그림 + 결과 표 저장
```

**v1 → v2 에서 바꾼 것** — v1 (PBE 포텐셜)은 LiFePO₄ ↔ FePO₄ 의 ΔV 를 1.8 % 로 냈다 (MP 6.4 %, 실험 약 6.5 %). 포텐셜 4종을 비교한 결과 **M3GNet r2SCAN 이 Fe 6.2 % · Mn 6.7 % 로 MP(6.4 · 6.6 %)를 재현**했고, Co · Ni 는 네 포텐셜 모두 MP 보다 크게 냈다. 그래서 **포텐셜은 M3GNet r2SCAN, 설계 범위는 검증을 통과한 Fe–Mn 계**로 좁혔다.

**규칙** — 1) 아는 물질로 먼저 재현한다 2) 검증을 통과한 범위에서만 설계한다 3) 최적화 결과엔 steps · fmax · 수렴 여부를 함께 적는다 4) comprehension 없이 for + if + append.

**ΔV 정의** — 화학식 단위(P 1개)당 부피로 `ΔV (%) = (V_방전 − V_충전) / V_충전 × 100`. 양수 = 충전하면 줄어든다.

**API key** 는 실행할 때 입력창에 넣는다 — 코드에 적지 않으므로 GitHub 에 그대로 올려도 안전하다.
실행 시간: 구조 최적화 약 30번 (28원자 셀), CPU 10~20분. 결과는 Google Drive 에 저장되어 세션이 끊겨도 다시 계산하지 않는다.

In [ ]:
# 0. 라이브러리 설치 — 세션마다 한 번 (Colab). 설치 후 '런타임 → 세션 다시 시작' 이 필요할 수 있다
!pip install matgl pymatgen mp-api ase pandas matplotlib

In [ ]:
# 공통 준비 — 이 셀을 먼저 실행
import warnings; warnings.filterwarnings('ignore')
import os, sys, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matgl
from pymatgen.core import Structure, Lattice
from pymatgen.io.ase import AseAtomsAdaptor
from matgl.ext.ase import PESCalculator, Relaxer
from getpass import getpass

USE_DRIVE = True                     # Colab 이면 True — 결과(CIF · 표)를 Google Drive 에 저장해 세션이 끊겨도 남긴다
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    FOLDER = '/content/drive/MyDrive/project_olivine_r2scan'   # v1(PBE) 결과와 섞이지 않게 새 폴더
else:
    FOLDER = 'project_olivine_r2scan'
os.makedirs(FOLDER, exist_ok=True)
print('Python', sys.version.split()[0], '/ matgl', matgl.__version__, '/ 저장 폴더', FOLDER)

API_KEY = getpass('Materials Project API key: ')   # 입력 내용은 화면 · 노트북 파일에 남지 않는다

In [ ]:
# [matgl]  포텐셜 불러오기 — 09a 와 같은 M3GNet, 학습 정답만 r2SCAN (포텐셜 비교에서 Fe · Mn ΔV 를 가장 잘 재현)
print('사용 가능한 모델:', matgl.get_available_pretrained_models())

PES = 'M3GNet-PES-MatPES-r2SCAN-2025.2'                 # 에너지 + 힘 + 응력 (MatPES r2SCAN 데이터로 학습)
pot = matgl.load_model(PES)
calc = PESCalculator(pot)
relaxer = Relaxer(potential=pot, relax_cell=True)     # FIRE, 셀(격자)까지 움직인다
FMAX, STEPS = 0.05, 500                               # 보고서에 적을 설정
print('포텐셜 준비 완료:', PES, '/ fmax', FMAX, '/ steps', STEPS)

In [ ]:
# 도구 함수 — 자리 찾기 · 부피 · 최적화(저장/불러오기) · Li 빼기 · 용량
REDOX = ['Fe', 'Mn', 'Co', 'Ni']                  # 2+ ↔ 3+ 산화환원을 하는 원소 (Mg 는 안 한다)

def sites_of(st, symbol):
    """symbol 원자의 자리 번호 목록"""
    idx = []
    for i, site in enumerate(st):
        if site.specie.symbol == symbol:
            idx.append(i)
    return idx

def count_redox(st):
    """산화환원 가능한 전이금속 개수 = 뺄 수 있는 Li 개수"""
    n = 0
    for el in REDOX:
        n += len(sites_of(st, el))
    return n

def vol_per_fu(st):
    """화학식 단위(P 1개)당 부피 (Å³) — 셀 크기가 달라도 비교할 수 있게"""
    return st.volume / len(sites_of(st, 'P'))

def delta_v(dis, chg):
    """ΔV (%) = (V_방전 − V_충전) / V_충전 × 100"""
    return (vol_per_fu(dis) - vol_per_fu(chg)) / vol_per_fu(chg) * 100

def capacity(dis, n_removed):
    """이론 용량 (mAh/g) = 26801 × (화학식당 뺀 Li 수) / (방전 상태 몰질량) — Ch.5"""
    n_fu = len(sites_of(dis, 'P'))
    return 26801 * (n_removed / n_fu) / (dis.composition.weight / n_fu)

def relax_or_load(st, name):
    """최적화 → CIF · 정보 저장. 이미 저장돼 있으면 다시 계산하지 않고 불러온다 (Colab 세션 대비)"""
    cif = f'{FOLDER}/{name}_relaxed.cif'
    info_path = f'{FOLDER}/{name}_info.json'
    if os.path.exists(cif) and os.path.exists(info_path):
        with open(info_path) as f:
            info = json.load(f)
        print(f'{name:22s} 불러옴 (저장된 결과)')
        return Structure.from_file(cif), info
    t0 = time.time()
    r = relaxer.relax(st.copy(), fmax=FMAX, steps=STEPS, verbose=False)
    fin = r['final_structure']
    traj = r['trajectory']
    fmax_last = float(np.abs(traj.forces[-1]).max())
    info = {'name': name, 'energy': float(traj.energies[-1]), 'steps': len(traj.energies),
            'fmax': round(fmax_last, 4), 'converged': bool(fmax_last < FMAX), 'sec': round(time.time() - t0)}
    fin.to(filename=cif)
    with open(info_path, 'w') as f:
        json.dump(info, f)
    print(f"{name:22s} {info['steps']:4d} 스텝, fmax {fmax_last:.3f}, {'수렴' if info['converged'] else '수렴 안 됨'}, {info['sec']} 초")
    return fin, info

def delithiate(st, n_remove=None):
    """충전 상태 = Li 를 n_remove 개 뺀 구조 (None 이면 전부).
    Mg 가 있으면 Mg 에서 먼 Li 부터 뺀다 — Mg 는 산화환원을 안 하므로 그만큼 Li 가 남아야 전하가 맞는다"""
    li = sites_of(st, 'Li')
    remove = []
    if n_remove is None or n_remove >= len(li):
        remove = li
    else:
        mg = sites_of(st, 'Mg')
        ranked = []
        for i in li:
            d_min = 99.0
            for j in mg:
                d_min = min(d_min, st.get_distance(i, j))
            ranked.append((d_min, i))
        ranked.sort(reverse=True)                     # Mg 에서 먼 Li 가 앞
        for d_min, i in ranked[:n_remove]:
            remove.append(i)
    chg = st.copy()
    chg.remove_sites(remove)
    return chg

# **단계 1 — [관문] LiFePO₄ 를 재현하고, Li 를 빼서 FePO₄ 의 부피 변화를 잰다**

---

```
MP LiFePO₄ (28원자, Li 4 · Fe 4) → 최적화 → 격자상수 vs 실험 · MP
→ Li 4개 전부 제거 (remove_sites) → FePO₄ 최적화 → ΔV
```

**판정** — 격자상수가 실험 대비 1~2 % 안이고, ΔV 가 실험(약 6.5 %) · MP(6.4 %) 와 가까우면 통과. (v1 의 PBE 포텐셜은 여기서 1.8 % 로 탈락했다)

In [ ]:
# 1. LiFePO₄ 모체 — MP 에서 받거나(실패하면 09b 의 실험 격자) → 최적화
from mp_api.client import MPRester

def get_mp(formula):
    '''화학식 → (가장 안정한 구조, material_id) — 09a 와 같은 함수'''
    with MPRester(API_KEY) as mpr:
        found = mpr.materials.summary.search(formula=formula, fields=['material_id', 'energy_above_hull', 'structure'])
    best = None
    for d in found:
        if best is None or d.energy_above_hull < best.energy_above_hull:
            best = d
    if best is None:
        raise ValueError(f'{formula} 를 MP 에서 찾지 못했습니다')
    return best.structure, str(best.material_id)

def make_lfp():
    '''실험 격자 LiFePO₄ (09b 와 같은 함수)'''
    return Structure.from_spacegroup('Pnma', Lattice.orthorhombic(10.33, 6.01, 4.69), ['Li', 'Fe', 'P', 'O', 'O', 'O'],
        [[0, 0, 0], [0.282, 0.25, 0.975], [0.095, 0.25, 0.418], [0.097, 0.25, 0.743], [0.457, 0.25, 0.206], [0.165, 0.047, 0.285]])

try:
    lfp_start, lfp_id = get_mp('LiFePO4')
    print('MP 에서 받음:', lfp_id)
except Exception as e:
    lfp_start, lfp_id = make_lfp(), '실험 격자'
    print('MP 없이 실험 격자로 시작 (사유:', type(e).__name__, ')')
print('원자', len(lfp_start), '개, Li', len(sites_of(lfp_start, 'Li')), '/ Fe', len(sites_of(lfp_start, 'Fe')))

lfp, info_lfp = relax_or_load(lfp_start, 'LiFePO4')

In [ ]:
# 2. 격자상수 재현 표 — 축 순서가 MP 와 실험이 다를 수 있어 작은 것부터 정렬해 비교
EXP_LFP = sorted([10.33, 6.01, 4.69])            # 실험 LiFePO₄ (Å)
start_abc = sorted(lfp_start.lattice.abc)
final_abc = sorted(lfp.lattice.abc)
rows = []
for k in range(3):
    rows.append({'축': ['짧은 축', '중간 축', '긴 축'][k], 'MP (시작)': round(start_abc[k], 3), '최적화 후': round(final_abc[k], 3),
                 '실험': EXP_LFP[k], '실험 대비 %': round((final_abc[k] / EXP_LFP[k] - 1) * 100, 1)})
df_lattice = pd.DataFrame(rows)
df_lattice.to_csv(FOLDER + '/lattice_LiFePO4.csv', index=False)
df_lattice
# 읽는 법: +0.5 ~ +2 % 면 정상 (PBE 라벨의 버릇, 09a 와 같다)

In [ ]:
# 3. 충전 상태 FePO₄ — 최적화된 LiFePO₄ 에서 Li 4개를 전부 빼고 다시 최적화
fp, info_fp = relax_or_load(delithiate(lfp), 'FePO4')
dv_lfp = delta_v(lfp, fp)
print(f'화학식당 부피: LiFePO₄ {vol_per_fu(lfp):.2f} → FePO₄ {vol_per_fu(fp):.2f} Å³')
print(f'ΔV (계산) = {dv_lfp:.1f} %   ← 실험 약 6.5 %, MP 6.4 % (v1 PBE 포텐셜은 1.8 %)')
print(f'이론 용량 = {capacity(lfp, 4):.0f} mAh/g  ← 손계산 26801 × 1 / 157.8 = 170')

# **단계 2 — MP 의 정답: 전극 데이터와 MP 구조로 잰 ΔV**

---

`insertion_electrodes` 의 `max_delta_volume` (Ch.6 사례 1 의 N4 조건) 과, 그 전극의 방전 · 충전 구조(`id_discharge` · `id_charge`)를 받아 **위와 같은 식**으로 직접 잰 ΔV 를 둘 다 적는다.
필드의 정의(어느 부피로 나누는가)가 내 식과 조금 다를 수 있어서, 같은 식으로 잰 값이 '사과 대 사과' 비교다.

In [ ]:
# [API key 필요]  MP 전극 항목: LiMPO₄ ↔ MPO₄
def mp_electrode(M):
    '''M → (max_delta_volume %, MP 구조로 잰 ΔV %, 방전 ↔ 충전 id). 없으면 None'''
    with MPRester(API_KEY) as mpr:
        docs = mpr.materials.insertion_electrodes.search(
            working_ion='Li', elements=[M, 'P', 'O'],
            fields=['formula_discharge', 'formula_charge', 'id_discharge', 'id_charge',
                    'max_delta_volume', 'stability_discharge'])
        best = None
        for d in docs:
            if d.formula_discharge == f'Li{M}PO4' and d.formula_charge == f'{M}PO4':
                s = d.stability_discharge if d.stability_discharge is not None else 99
                if best is None or s < best[0]:
                    best = (s, d)
        if best is None:
            return None, None, None
        d = best[1]
        s_dis = mpr.get_structure_by_material_id(str(d.id_discharge))
        s_chg = mpr.get_structure_by_material_id(str(d.id_charge))
    return round(d.max_delta_volume * 100, 1), round(delta_v(s_dis, s_chg), 1), f'{d.id_discharge} ↔ {d.id_charge}'

MP_DV = {}
for M in ['Fe', 'Mn', 'Co', 'Ni']:
    try:
        field, from_st, ids = mp_electrode(M)
    except Exception as e:
        field, from_st, ids = None, None, None
        print(M, 'MP 조회 실패:', e)
    MP_DV[M] = {'field': field, 'from_structure': from_st}
    print(f'Li{M}PO4 ↔ {M}PO4   max_delta_volume {field} %   MP 구조로 잰 ΔV {from_st} %   ({ids})')
# None 이 나온 원소는 MP 에 해당 전극 항목이 없는 것 — 패리티에서 빼고 슬라이드에 'DB 값 없음' 으로 적는다

# **단계 3 — [검증] 전부 치환 LiMPO₄ 의 ΔV 를 MP 와 대조한다**

---

```
최적화된 LiFePO₄ → Fe 4개를 전부 M 으로 → 최적화 (LiMPO₄) → Li 전부 제거 → 최적화 (MPO₄) → ΔV
```

전부 치환한 구조는 MP 에 정답이 있으므로 **검증**이다 (09b 단계 4 의 '전부 치환 = 검증, 절반 치환 = 설계' 논리).

In [ ]:
# 전부 치환 3종 + LiFePO₄ (단계 1 결과) → 표
fe_sites = sites_of(lfp, 'Fe')
results = []
results.append({'구조': 'LiFePO4', '치환': 'Fe (기준)', '비율 %': 0, 'ΔV 계산 %': round(dv_lfp, 2),
                'MP field %': MP_DV['Fe']['field'], 'MP 구조 %': MP_DV['Fe']['from_structure'],
                '용량 mAh/g': round(capacity(lfp, 4)), '수렴': info_lfp['converged'] and info_fp['converged']})
for M in ['Mn', 'Co', 'Ni']:
    sub = lfp.copy()
    for i in fe_sites:
        sub[i] = M
    dis, i1 = relax_or_load(sub, f'Li{M}PO4')
    chg, i2 = relax_or_load(delithiate(dis), f'{M}PO4')
    results.append({'구조': f'Li{M}PO4', '치환': M, '비율 %': 100, 'ΔV 계산 %': round(delta_v(dis, chg), 2),
                    'MP field %': MP_DV[M]['field'], 'MP 구조 %': MP_DV[M]['from_structure'],
                    '용량 mAh/g': round(capacity(dis, 4)), '수렴': i1['converged'] and i2['converged']})
df_full = pd.DataFrame(results)
df_full.to_csv(FOLDER + '/results_all.csv', index=False)
df_full

In [ ]:
# 패리티 그림 — 계산 ΔV vs MP max_delta_volume (점이 대각선 위에 있으면 재현)
xs = []; ys = []; labels = []
for r in results:
    if r['MP field %'] is not None:
        xs.append(r['MP field %']); ys.append(r['ΔV 계산 %']); labels.append(r['구조'])
plt.figure(figsize=(4, 4))
plt.scatter(xs, ys, s=60, color='#002060', zorder=3)
for x, y, lab in zip(xs, ys, labels):
    plt.annotate(lab, (x, y), textcoords='offset points', xytext=(5, 5), fontsize=9)
if len(xs) > 0:
    lo = min(min(xs), min(ys)) - 1; hi = max(max(xs), max(ys)) + 1
    plt.plot([lo, hi], [lo, hi], '--', color='gray')
    mae = np.mean(np.abs(np.array(xs) - np.array(ys)))
    plt.title(f'Volume change: potential vs MP (MAE {mae:.1f} %)', fontsize=10)
plt.xlabel('MP (DFT+U) dV (%)'); plt.ylabel('M3GNet-PES r2SCAN dV (%)')
plt.tight_layout(); plt.savefig(FOLDER + '/fig_parity.png', dpi=200); plt.show()
# 읽는 법: Fe · Mn 은 대각선 위(재현), Co · Ni 는 위로 벗어난다(과대) → 설계는 Fe–Mn 계에서만

# **참고 — 포텐셜 4종 비교 (v1 에서 계산한 결과를 불러와 그림으로)**

---

같은 8개 구조(LiMPO₄ · MPO₄, M = Fe · Mn · Co · Ni)를 포텐셜 네 개로 각각 최적화해 ΔV 를 MP 와 비교한 표. 계산은 v1 노트북에서 끝났으므로 저장된 표를 불러온다 (다시 계산하려면 `RUN_BENCH = True`, 10~20분).

In [ ]:
# 포텐셜 비교 표 불러오기 → 그림 (슬라이드 '검증')
RUN_BENCH = False
BENCH_CSV = '/content/drive/MyDrive/project_olivine/potential_benchmark.csv'   # v1 이 저장한 곳
CANDIDATES = ['M3GNet-PES-MatPES-PBE-2025.2', 'M3GNet-PES-MatPES-r2SCAN-2025.2',
              'CHGNet-PES-MatPES-r2SCAN-2025.2.10', 'TensorNet-PES-MatPES-r2SCAN-2025.2']
MP_REF = {'Fe': 6.4, 'Mn': 6.6, 'Co': 4.1, 'Ni': 3.6}

if RUN_BENCH:
    def relax_cached(rlx, st, path):
        if os.path.exists(path + '.cif'):
            return Structure.from_file(path + '.cif')
        r = rlx.relax(st.copy(), fmax=FMAX, steps=STEPS, verbose=False)
        fin = r['final_structure']
        fin.to(filename=path + '.cif')
        return fin
    bench_rows = []
    for pes_name in CANDIDATES:
        relaxer_b = Relaxer(potential=matgl.load_model(pes_name), relax_cell=True)
        sub = f'{FOLDER}/bench_{pes_name}'
        os.makedirs(sub, exist_ok=True)
        row = {'포텐셜': pes_name}
        errs = []
        for M in ['Fe', 'Mn', 'Co', 'Ni']:
            st = lfp_start.copy()
            for i in sites_of(st, 'Fe'):
                st[i] = M
            dis = relax_cached(relaxer_b, st, f'{sub}/Li{M}PO4')
            chg = relax_cached(relaxer_b, delithiate(dis), f'{sub}/{M}PO4')
            row[M] = round(delta_v(dis, chg), 1)
            errs.append(abs(row[M] - MP_REF[M]))
        row['MAE %'] = round(float(np.mean(errs)), 1)
        bench_rows.append(row)
    row_mp = {'포텐셜': 'MP (DFT+U, 정답)'}
    for M in MP_REF:
        row_mp[M] = MP_REF[M]
    row_mp['MAE %'] = 0.0
    bench_rows.append(row_mp)
    df_bench = pd.DataFrame(bench_rows)
    BENCH_CSV = FOLDER + '/potential_benchmark.csv'
    df_bench.to_csv(BENCH_CSV, index=False)

if os.path.exists(BENCH_CSV):
    df_bench = pd.read_csv(BENCH_CSV)
    # 그림: 원소별 ΔV — 포텐셜 4종 막대 + MP 정답 (검은 가로선)
    els = ['Fe', 'Mn', 'Co', 'Ni']
    short = {'M3GNet-PES-MatPES-PBE-2025.2': 'M3GNet PBE', 'M3GNet-PES-MatPES-r2SCAN-2025.2': 'M3GNet r2SCAN',
             'CHGNet-PES-MatPES-r2SCAN-2025.2.10': 'CHGNet r2SCAN', 'TensorNet-PES-MatPES-r2SCAN-2025.2': 'TensorNet r2SCAN'}
    colors = ['#BFBFBF', '#002060', '#9DC3E6', '#4472C4']
    plt.figure(figsize=(7, 3.5))
    w = 0.2
    k = 0
    for _, r in df_bench.iterrows():
        if r['포텐셜'] in short:
            xs = []
            for n in range(len(els)):
                xs.append(n + (k - 1.5) * w)
            vals = []
            for el in els:
                vals.append(r[el])
            plt.bar(xs, vals, width=w, color=colors[k], label=f"{short[r['포텐셜']]} (MAE {r['MAE %']})")
            k += 1
    for n, el in enumerate(els):
        plt.hlines(MP_REF[el], n - 0.45, n + 0.45, color='k', lw=2)
    plt.xticks(range(len(els)), ['LiFePO4', 'LiMnPO4', 'LiCoPO4', 'LiNiPO4'])
    plt.ylabel('dV on charge (%)'); plt.title('Which potential reproduces MP (black line)?', fontsize=10)
    plt.legend(fontsize=7); plt.tight_layout(); plt.savefig(FOLDER + '/fig_benchmark.png', dpi=200); plt.show()
    display(df_bench)
else:
    print('포텐셜 비교 표가 없다 — v1 노트북에서 계산했는지, 또는 RUN_BENCH = True 로 다시 계산')
# 읽는 법: Fe · Mn 에서 M3GNet r2SCAN 막대만 검은 선에 닿는다. Co · Ni 는 네 포텐셜 모두 위로 벗어난다

# **단계 4 — [설계] LMFP: Mn 비율 0 ~ 100 % 와 Mg 25 % 도핑**

---

```
LiFePO₄ (전이금속 자리 4개) → Mn 1 · 2 · 3 · 4 개 = 25 · 50 · 75 · 100 %
                            → 50 % 는 가까운 배치 · 먼 배치 둘 다 최적화해 에너지가 낮은 쪽 (09b 단계 5)
Mg 25 % : LiFePO₄ 의 Fe 1개 → Mg,  LMFP 50 % 의 남은 Fe 1개 → Mg
→ 충전 상태: 산화환원 가능한 전이금속(Fe · Mn) 수만큼 Li 를 뺀다
```

**왜 Fe–Mn 만** — 단계 3 · 포텐셜 비교에서 Fe · Mn 만 MP 를 재현했다. 검증을 통과한 범위에서만 설계한다.
**Mg 는 산화환원을 안 한다** — Mg 25 % 면 Li 4개 중 3개만 뺄 수 있다 → 부피 변화와 함께 용량도 준다.

In [ ]:
# [설계] LMFP — 셀 하나에서 Mn 0 · 25 · 50 · 75 · 100 % + Mg 25 % (LFP, LMFP 50 % 각각)
fe_sites = sites_of(lfp, 'Fe')
i0 = fe_sites[0]
dists = []
for j in fe_sites[1:]:
    dists.append((lfp.get_distance(i0, j), j))
dists.sort()
j_near, j_far = dists[0][1], dists[-1][1]
print(f'첫 Fe 에서 가장 가까운 Fe {dists[0][0]:.2f} Å, 가장 먼 Fe {dists[-1][0]:.2f} Å')

def run_case(st, name, x_mn, mg):
    '''방전 상태 최적화 → Fe · Mn 수만큼 Li 제거 → 충전 상태 최적화 → 한 줄'''
    dis, i1 = relax_or_load(st, name)
    n_rm = count_redox(dis)
    chg, i2 = relax_or_load(delithiate(dis, n_rm), name + '_chg')
    return {'구조': dis.composition.reduced_formula, 'Mn %': x_mn, 'Mg %': mg,
            'ΔV 계산 %': round(delta_v(dis, chg), 2), '용량 mAh/g': round(capacity(dis, n_rm)),
            '수렴': i1['converged'] and i2['converged']}

design = []
# Mn 0 % (LiFePO4) — 단계 1 결과 재사용
design.append({'구조': 'LiFePO4', 'Mn %': 0, 'Mg %': 0, 'ΔV 계산 %': round(dv_lfp, 2),
               '용량 mAh/g': round(capacity(lfp, 4)), '수렴': info_lfp['converged'] and info_fp['converged']})
# Mn 25 %
s = lfp.copy(); s[i0] = 'Mn'
design.append(run_case(s, 'Mn25', 25, 0))
# Mn 50 % — 두 배치 중 방전 상태 에너지가 낮은 쪽
cfg = []
for lab, j in [('near', j_near), ('far', j_far)]:
    s = lfp.copy(); s[i0] = 'Mn'; s[j] = 'Mn'
    d50, inf = relax_or_load(s, f'Mn50_{lab}')
    cfg.append((inf['energy'], lab, j))
    print(f'  Mn 50 % {lab:4s} 배치 에너지 {inf["energy"]:.3f} eV')
cfg.sort()
lab50, j50 = cfg[0][1], cfg[0][2]
print(f'  → 낮은 쪽: {lab50} (차이 {cfg[1][0] - cfg[0][0]:.3f} eV/셀)')
s = lfp.copy(); s[i0] = 'Mn'; s[j50] = 'Mn'
design.append(run_case(s, f'Mn50_{lab50}', 50, 0))
# Mn 75 % — Fe 1개만 남긴다
s = lfp.copy()
for j in fe_sites[1:]:
    s[j] = 'Mn'
design.append(run_case(s, 'Mn75', 75, 0))
# Mn 100 % (LiMnPO4)
s = lfp.copy()
for j in fe_sites:
    s[j] = 'Mn'
design.append(run_case(s, 'Mn100', 100, 0))
# Mg 25 % — LiFePO4 에
s = lfp.copy(); s[i0] = 'Mg'
design.append(run_case(s, 'Mg25_Mn0', 0, 25))
# Mg 25 % — LMFP 50 % 의 남은 Fe 하나에
s = lfp.copy(); s[i0] = 'Mn'; s[j50] = 'Mn'
left = []
for j in fe_sites:
    if j != i0 and j != j50:
        left.append(j)
s[left[0]] = 'Mg'
design.append(run_case(s, 'Mg25_Mn50', 50, 25))

df_design = pd.DataFrame(design)
df_design.to_csv(FOLDER + '/results_design.csv', index=False)
df_design

# **단계 5 — 그림 두 장: Mn 비율에 따른 ΔV · 용량, 그리고 Mg 의 효과**

---

In [ ]:
# 그림 1 — Mn 비율 vs ΔV (Mg 없는 것 선, Mg 25 % 는 빨간 점) (슬라이드 '결과')
x_line = []; y_line = []; x_mg = []; y_mg = []
for r in design:
    if r['Mg %'] == 0:
        x_line.append(r['Mn %']); y_line.append(r['ΔV 계산 %'])
    else:
        x_mg.append(r['Mn %']); y_mg.append(r['ΔV 계산 %'])
plt.figure(figsize=(5, 3.5))
plt.plot(x_line, y_line, 'o-', color='#002060', label='LiMn$_x$Fe$_{1-x}$PO$_4$')
plt.scatter(x_mg, y_mg, s=70, marker='v', color='#C00000', zorder=3, label='+ Mg 25 %')
plt.axhline(MP_REF['Fe'], ls=':', color='gray', lw=1); plt.axhline(MP_REF['Mn'], ls=':', color='gray', lw=1)
plt.xlabel('Mn fraction (%)'); plt.ylabel('dV on charge (%)'); plt.xticks([0, 25, 50, 75, 100])
plt.title('Volume change vs Mn content (M3GNet r2SCAN)', fontsize=10); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FOLDER + '/fig_design_dV.png', dpi=200); plt.show()
# 점선: MP 의 LiFePO4 (6.4 %) · LiMnPO4 (6.6 %)

In [ ]:
# 그림 2 — ΔV vs 이론 용량: 왼쪽 위(작은 ΔV, 큰 용량)가 좋다 (슬라이드 '해석')
plt.figure(figsize=(5, 3.5))
for r in design:
    c = '#C00000' if r['Mg %'] > 0 else '#002060'
    m = 'v' if r['Mg %'] > 0 else 'o'
    plt.scatter(r['ΔV 계산 %'], r['용량 mAh/g'], s=60, color=c, marker=m)
    lab = f"Mn {r['Mn %']}%" + (' +Mg' if r['Mg %'] > 0 else '')
    plt.annotate(lab, (r['ΔV 계산 %'], r['용량 mAh/g']), textcoords='offset points', xytext=(5, 3), fontsize=8)
plt.xlabel('dV on charge (%)'); plt.ylabel('theoretical capacity (mAh/g)')
plt.title('Trade-off: volume change vs capacity', fontsize=10)
plt.tight_layout(); plt.savefig(FOLDER + '/fig_tradeoff.png', dpi=200); plt.show()

# **확인 · 발표용 그림 — Mg 배치 확인, 슬라이드 그림 두 장**

---

In [ ]:
# [확인] Mg 결과가 배치 하나의 우연인가 — 다른 배치로 한 번 더
def delithiate_keep_far(st, n_remove):
    # delithiate 의 반대: Mg 에 가까운 Li 부터 뺀다 (남는 Li 가 Mg 에서 멀다)
    li = sites_of(st, 'Li'); mg = sites_of(st, 'Mg')
    ranked = []
    for i in li:
        d_min = 99.0
        for j in mg:
            d_min = min(d_min, st.get_distance(i, j))
        ranked.append((d_min, i))
    ranked.sort()
    remove = []
    for d_min, i in ranked[:n_remove]:
        remove.append(i)
    chg = st.copy(); chg.remove_sites(remove)
    return chg

def load_info(name):
    with open(f'{FOLDER}/{name}_info.json') as f:
        return json.load(f)

# (a) LFP + Mg — 충전 상태에서 남길 Li 를 반대로
s = lfp.copy(); s[i0] = 'Mg'
dis_a, _ = relax_or_load(s, 'Mg25_Mn0')
chg_a, ia = relax_or_load(delithiate_keep_far(dis_a, count_redox(dis_a)), 'Mg25_Mn0_chgFar')
e_near = load_info('Mg25_Mn0_chg')['energy']
print(f"(a) LFP+Mg  남은 Li 가 Mg 옆: ΔV {design[5]['ΔV 계산 %']:.2f} % (E {e_near:.3f})   "
      f"Mg 에서 멀리: ΔV {delta_v(dis_a, chg_a):.2f} % (E {ia['energy']:.3f})")

# (b) LMFP 50 % + Mg — Mg 를 남은 다른 Fe 자리에
s = lfp.copy(); s[i0] = 'Mn'; s[j50] = 'Mn'; s[left[1]] = 'Mg'
row_b = run_case(s, 'Mg25_Mn50_siteB', 50, 25)
e_a = load_info('Mg25_Mn50')['energy']; e_b = load_info('Mg25_Mn50_siteB')['energy']
print(f"(b) LMFP+Mg 자리 A: ΔV {design[6]['ΔV 계산 %']:.2f} % (E {e_a:.3f})   "
      f"자리 B: ΔV {row_b['ΔV 계산 %']:.2f} % (E {e_b:.3f})")
# 결과: (a) 5.16 vs 5.18 % — Li 배치 무관. (b) 두 자리가 에너지까지 같다 = 대칭상 같은 자리

In [ ]:
# 발표용 그림 1 — 섞으면 커진다 + Mg 화살표
x0 = []; y0 = []; mg_pts = []; base = {}
for r in design:
    if r['Mg %'] == 0:
        x0.append(r['Mn %']); y0.append(r['ΔV 계산 %']); base[r['Mn %']] = r['ΔV 계산 %']
    else:
        mg_pts.append((r['Mn %'], r['ΔV 계산 %']))
plt.figure(figsize=(6, 4))
plt.plot([0, 100], [base[0], base[100]], '--', color='#7F7F7F', lw=1.5, label='linear mix of end members')
plt.plot(x0, y0, 'o-', color='#002060', lw=2, ms=8, label='LiMn$_x$Fe$_{1-x}$PO$_4$ (calc.)')
plt.scatter([0, 100], [MP_REF['Fe'], MP_REF['Mn']], s=90, facecolors='none', edgecolors='k', label='MP (DFT+U)', zorder=4)
for k, (x, y) in enumerate(mg_pts):
    plt.annotate('', xy=(x, y), xytext=(x, base[x]), arrowprops=dict(arrowstyle='->', color='#C00000', lw=2))
    plt.scatter(x, y, s=90, marker='v', color='#C00000', zorder=4, label='+ Mg 25 %' if k == 0 else None)
plt.xlabel('Mn fraction (%)', fontsize=12); plt.ylabel('volume change on charge (%)', fontsize=12)
plt.xticks([0, 25, 50, 75, 100]); plt.legend(fontsize=9, loc='lower right')
plt.tight_layout(); plt.savefig(FOLDER + '/slide_fig1_mixing.png', dpi=300); plt.show()

# 발표용 그림 2 — Mg 효과: ΔV 와 용량을 나란히
picks = [('LFP', 0, 0), ('LFP\n+ Mg', 0, 25), ('LMFP 50%', 50, 0), ('LMFP 50%\n+ Mg', 50, 25)]
names = []; dvs = []; caps = []; cols = []
for name, mn, mg in picks:
    for r in design:
        if r['Mn %'] == mn and r['Mg %'] == mg:
            names.append(name); dvs.append(r['ΔV 계산 %']); caps.append(r['용량 mAh/g'])
            cols.append('#C00000' if mg > 0 else '#002060')
fig, ax = plt.subplots(1, 2, figsize=(8, 4.2))
b0 = ax[0].bar(names, dvs, color=cols); ax[0].set_ylabel('volume change (%)', fontsize=11)
b1 = ax[1].bar(names, caps, color=cols); ax[1].set_ylabel('capacity (mAh/g)', fontsize=11)
for b, v in zip(b0, dvs):
    ax[0].text(b.get_x() + b.get_width() / 2, v + 0.1, f'{v:.1f}', ha='center', fontsize=10)
for b, v in zip(b1, caps):
    ax[1].text(b.get_x() + b.get_width() / 2, v + 2, f'{v}', ha='center', fontsize=10)
ax[0].set_ylim(0, max(dvs) * 1.15); ax[1].set_ylim(0, 190)
ax[0].set_title('smaller is better', fontsize=10); ax[1].set_title('larger is better', fontsize=10)
plt.tight_layout(); plt.savefig(FOLDER + '/slide_fig2_mg.png', dpi=300); plt.show()

# **마무리 — 결과 읽는 법 · 발표 · 한계**

---

| 숫자 | 이렇게 읽는다 |
|---|---|
| LiFePO₄ 격자 | 실험 대비 1~2 % 안이면 정상 |
| LiFePO₄ ΔV | MP 6.4 % · 실험 약 6.5 % 와 가까우면 관문 통과 (v1 PBE 포텐셜은 1.8 %) |
| 패리티 · 포텐셜 비교 | Fe · Mn 만 재현 → 설계를 Fe–Mn 계로 한정한 근거 |
| Mn 비율 vs ΔV | 0 → 100 % 사이가 직선(두 끝의 평균)인지, 중간 조성이 더 작거나 큰지 |
| Mg 25 % | ΔV 가 줄면 용량도 함께 준다 (산화환원 안 함) — 산점도에서 대가를 같이 보인다 |
| 수렴 False | steps 를 늘리기 전에 시작 구조를 의심 (09a 에러 표) |

### 한계 (발표 '한계' 슬라이드)
1. Co · Ni 계는 네 포텐셜 모두 MP 보다 ΔV 를 크게 냈다 — 원인은 확인하지 못했고, 그래서 설계에서 뺐다
2. 포텐셜은 전하 · 자성을 직접 보지 않는다 — 그래도 r2SCAN 라벨에서 Fe · Mn 의 부피 변화는 재현됐다
3. 끝점 두 상태(완전 방전 · 충전)만 본다 — 중간 상, 축마다 다른 변형, 상전이는 못 본다
4. 28원자 셀 — 조성 간격 25 %, 배치는 일부만 계산 (50 % 는 두 배치 중 낮은 쪽)
5. 0 K, 완벽한 결정. 전압(에너지밀도)은 계산하지 않았다

### 보고서 · GitHub 에 적을 것
- 포텐셜 `M3GNet-PES-MatPES-r2SCAN-2025.2`, matgl 버전, fmax 0.05, steps 500, relax_cell=True
- `results_design.csv`, `results_all.csv`, `lattice_LiFePO4.csv`, 그림 (`FOLDER` 에 저장됨)
- **API key 는 코드에 없다** (getpass) — 그대로 공개 저장소에 올려도 된다